# Section 4: Model Selection

This notebook uses the preprocessing and Logistic Regression model from Feature Selection, with the selected Decision Tree feature selector. Logistic Regression turns a weighted sum of the selected features into a class probability. `C` controls regularisation; `C=1` is the baseline, and the other values are compared on the same chronological validation folds with a 90-day gap.

In [29]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier

# Find the project data folder whether the notebook runs from the root or notebooks/.
project_folder = Path.cwd()
if not (project_folder / "data" / "train.csv").is_file():
    project_folder = project_folder.parent
if not (project_folder / "data" / "train.csv").is_file():
    raise FileNotFoundError("Could not find data/train.csv from the notebook's working directory.")

data_folder = project_folder / "data"
train = pd.read_csv(data_folder / "train.csv", index_col="ID")
test = pd.read_csv(data_folder / "test.csv", index_col="ID")
sample_submission = pd.read_csv(data_folder / "sample_submission.csv")

TARGET = "repeat_purchase_90d"
RANDOM_STATE = 42
assert TARGET in train.columns and TARGET not in test.columns
assert train.index.is_unique and test.index.is_unique
assert train[TARGET].notna().all()
assert set(train[TARGET].unique()) == {0, 1}
assert set(test.columns) == set(train.drop(columns=TARGET).columns)

X = train.drop(columns=TARGET)
y = train[TARGET].astype(int)

print(f"Training rows: {len(train):,}; test rows: {len(test):,}")
display(y.value_counts(normalize=True).sort_index().rename("target proportion").to_frame())

Training rows: 6,534; test rows: 986


,target proportion
repeat_purchase_90d,
0,0.440771
1,0.559229


In [30]:
# Split unique dates into chronological blocks; the last four are validation windows.
purchase_dates = pd.to_datetime(X["purchase_date"], errors="coerce")
unique_dates = pd.Index(purchase_dates.dropna().unique()).sort_values()
date_blocks = np.array_split(unique_dates.to_numpy(), 10)
validation_blocks = date_blocks[6:]
assert len(validation_blocks) == 4 and all(len(block) > 0 for block in validation_blocks)

print(f"Validation windows: {len(validation_blocks)}")
print(f"Rows without a purchase date: {purchase_dates.isna().sum():,}")

Validation windows: 4
Rows without a purchase date: 65


In [31]:
# Match Feature Selection's columns and preparation steps.
categorical_columns = [
    "payment_type", "salesperson", "commercial_zone",
    "transport_method", "warehouse", "document_series",
]
numeric_columns = X.select_dtypes(include="number").columns.tolist()
numeric_columns.remove("customer_id")
numeric_columns.remove("random_noise")


def clean_values(data):
    cleaned = data.copy()
    # Treat impossible values and known sentinel values as missing.
    for column in numeric_columns:
        cleaned[column] = cleaned[column].replace([np.inf, -np.inf], np.nan)
        cleaned.loc[cleaned[column] < 0, column] = np.nan
    for column in ["prior_mean_purchase_value", "spend_365d"]:
        cleaned.loc[cleaned[column] >= 1_000_000_000, column] = np.nan
    return cleaned


def add_features(data):
    prepared = clean_values(data)
    # Keep missingness as a signal before imputing the values.
    for column in numeric_columns:
        prepared[column + "_missing"] = prepared[column].isna().astype(int)

    prepared["first_purchase"] = (prepared["prior_purchase_count"] == 0).astype(int)
    prepared["history_unknown"] = prepared["prior_purchase_count"].isna().astype(int)
    # Divide only when the purchase has at least one detail.
    detail_count = prepared["N_ Detalhes"].where(prepared["N_ Detalhes"] > 0)
    prepared["net_value_per_detail"] = prepared["Total Liquido"] / detail_count

    for column in categorical_columns:
        # Trim labels and make empty strings consistent with missing values.
        labels = prepared[column].astype("string").str.strip().replace("", pd.NA)
        prepared[column] = labels.to_numpy(dtype=object, na_value=np.nan)

    return prepared.drop(columns=["customer_id", "purchase_date", "random_noise"])


log_columns = numeric_columns + ["net_value_per_detail"]


def prepare_features(data):
    prepared = add_features(data)
    # Use the same log transform for the numeric columns as Feature Selection.
    for column in log_columns:
        prepared[column] = np.log1p(prepared[column])
    return prepared


model_numeric_columns = prepare_features(X).select_dtypes(include="number").columns.tolist()

In [32]:
# Imputation, scaling, and encoding are fitted only on each fold's training rows.
def build_preprocessor(scale=True):
    numeric_steps = [("fill_missing", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)

    categorical_pipeline = Pipeline([
        ("fill_missing", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    combine_columns = ColumnTransformer([
        ("numeric", numeric_pipeline, model_numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ])
    return Pipeline([
        ("prepare_features", FunctionTransformer(prepare_features)),
        ("combine_columns", combine_columns),
    ])


def build_model(C=1.0):
    return Pipeline([
        ("preprocessing", build_preprocessor()),
        # Reuse the final tree-importance selector from Feature Selection.
        ("feature_selection", SelectFromModel(
            DecisionTreeClassifier(random_state=RANDOM_STATE), threshold="mean"
        )),
        ("model", LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE)),
    ])

In [33]:
# Compare regularization strengths on the same chronological folds.
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
fold_results = []
last_fold_models = {}

for fold_number, validation_dates in enumerate(validation_blocks, start=1):
    validation_start = pd.Timestamp(validation_dates[0])
    validation_end = pd.Timestamp(validation_dates[-1])
    # Leave 90 days between training and validation to respect the target horizon.
    fitting_end = validation_start - pd.Timedelta(days=90)
    fitting_rows = purchase_dates < fitting_end
    validation_rows = purchase_dates.between(validation_start, validation_end, inclusive="both")

    X_fit = X.loc[fitting_rows]
    y_fit = y.loc[fitting_rows]
    X_validation = X.loc[validation_rows]
    y_validation = y.loc[validation_rows]
    assert not X_fit.empty and not X_validation.empty
    assert y_fit.nunique() == 2 and y_validation.nunique() == 2

    # The pipeline refits preprocessing and feature selection within each fold.
    for C in C_values:
        model = build_model(C=C).fit(X_fit, y_fit)
        predictions = model.predict(X_validation)
        fold_results.append({
            "fold": fold_number,
            "C": C,
            "validation_start": validation_start.date(),
            "validation_end": validation_end.date(),
            "accuracy": accuracy_score(y_validation, predictions),
            "F1": f1_score(y_validation, predictions),
        })
        if fold_number == len(validation_blocks):
            last_fold_models[C] = model
            last_validation_rows = validation_rows

fold_results = pd.DataFrame(fold_results)
# Select by mean validation F1; accuracy is only used to break a tie.
results = (
    fold_results.groupby("C")
    .agg(mean_accuracy=("accuracy", "mean"), mean_F1=("F1", "mean"), std_F1=("F1", "std"))
    .sort_values(["mean_F1", "mean_accuracy"], ascending=False)
)
display(results.round(3))

best_C = float(results.index[0])
best_model = last_fold_models[best_C]
X_validation = X.loc[last_validation_rows]
y_validation = y.loc[last_validation_rows]
print(f"Selected C: {best_C}")

,mean_accuracy,mean_F1,std_F1
C,,,
0.100,0.737,0.762,0.030
1.000,0.736,0.760,0.033
10.000,0.736,0.760,0.035
100.000,0.736,0.759,0.034
0.010,0.725,0.755,0.031
0.001,0.709,0.755,0.033


Selected C: 0.1


In [34]:
# Show which prepared columns the tree selector retained in the latest fold.
feature_names = best_model.named_steps["preprocessing"].named_steps[
    "combine_columns"
].get_feature_names_out()
selected = best_model.named_steps["feature_selection"].get_support()
print(f"Selected {selected.sum()} of {len(selected)} prepared features")
display(pd.Series(feature_names[selected], name="selected feature").to_frame())

# Numeric inputs are scaled; categorical inputs are one-hot encoded.
coefficients = best_model.named_steps["model"].coef_[0]
coefficient_table = pd.DataFrame({
    "feature": feature_names[selected],
    "coefficient": coefficients,
})
print("Coefficients show each feature's direction in the model's log-odds, not causal effects.")
display(
    coefficient_table.assign(abs_coefficient=coefficient_table["coefficient"].abs())
    .nlargest(10, "abs_coefficient")
    .drop(columns="abs_coefficient")
    .reset_index(drop=True)
)

# Report final-fold performance and the confusion matrix for class 0 and class 1.
predictions = best_model.predict(X_validation)
print("Latest validation window (rows=true, columns=predicted):")
display(pd.DataFrame(
    confusion_matrix(y_validation, predictions, labels=[0, 1]),
    index=["actual 0", "actual 1"],
    columns=["predicted 0", "predicted 1"],
))
print(f"Accuracy: {accuracy_score(y_validation, predictions):.3f}")
print(f"Precision: {precision_score(y_validation, predictions):.3f}")
print(f"Recall: {recall_score(y_validation, predictions):.3f}")
print(f"F1: {f1_score(y_validation, predictions):.3f}")

Selected 28 of 169 prepared features


,selected feature
0,numeric__Total a Pagar
1,numeric__Total Bruto
2,numeric__Total Liquido
3,numeric__Total Imp_
4,numeric__Total Desc_ Linha
5,numeric__Custo
6,numeric__N_ Detalhes
7,numeric__days_since_previous_purchase
8,numeric__customer_tenure_days
9,numeric__prior_purchase_count


Coefficients show each feature's direction in the model's log-odds, not causal effects.


,feature,coefficient
0,numeric__purchase_count_365d,0.720943
1,categorical__salesperson_VLOOKUP Master 4,-0.403278
2,numeric__customer_tenure_days,-0.368298
3,numeric__spend_90d,0.332516
4,numeric__Total Imp_,0.307680
5,numeric__Total Desc_ Linha,0.298787
6,numeric__prior_total_spend,0.267709
7,numeric__purchase_count_180d,0.263739
8,categorical__payment_type_Maybe Tomorrow Payme...,-0.259664
9,numeric__spend_180d,-0.246170


Latest validation window (rows=true, columns=predicted):


,predicted 0,predicted 1
actual 0,164,72
actual 1,76,286


Accuracy: 0.753
Precision: 0.799
Recall: 0.790
F1: 0.794


In [35]:
# Refit the selected setup on all labelled rows before predicting the test set.
final_model = build_model(C=best_C).fit(X, y)
test_predictions = final_model.predict(test)

assert sample_submission.columns.tolist() == ["ID", TARGET]
assert sample_submission["ID"].equals(pd.Series(test.index, name="ID"))
assert len(test_predictions) == len(sample_submission)
assert set(np.unique(test_predictions)).issubset({0, 1})

submission = sample_submission.copy()
submission[TARGET] = test_predictions
submission_path = project_folder / "submission.csv"
# Keep the sample submission's ID order and column layout.
submission.to_csv(submission_path, index=False)
print(f"Wrote {len(submission):,} predictions to {submission_path}")
display(submission.head())

Wrote 986 predictions to /home/yoru/Documents/novaims/Y1/S1/ML/ML_Project/submission.csv


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1
